In [8]:
import numpy as np
import pandas as pd

# 1. Load the dataset directly from Google Drive link (no manual upload needed)
url = 'https://drive.google.com/uc?export=download&id=1JW_AK9i94nbLF5ris2vJjZA-kQV1KGMa'
df = pd.read_csv(url)

print('Shape before cleaning:', df.shape)
print('\nMissing values before cleaning:')
print(df.isna().sum())

# 2. Remove exact duplicate rows
df = df.drop_duplicates().copy()

# 3. Clean Quantity: remove 'pcs', convert to numeric, and fix negative values
df['Quantity'] = (
    df['Quantity'].astype(str).str.replace('pcs', '', case=False).str.strip()
)
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df.loc[df['Quantity'] <= 0, 'Quantity'] = np.nan
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median()).astype(int)

# 4. Clean Discount: convert percentages like '15%' to 0.15 and cap invalid values (> 1)
df['Discount'] = df['Discount'].astype(str).str.strip()
is_pct = df['Discount'].str.endswith('%')
df.loc[is_pct, 'Discount'] = (
    pd.to_numeric(df.loc[is_pct, 'Discount'].str.rstrip('%'), errors='coerce')
    / 100.0
)
df['Discount'] = pd.to_numeric(df['Discount'], errors='coerce')
df.loc[(df['Discount'] < 0) | (df['Discount'] > 1), 'Discount'] = np.nan
df['Discount'] = df['Discount'].fillna(0.0)

# 5. Clean Shipping_Cost & Unit_Price
for col in ['Unit_Price', 'Shipping_Cost']:
  df[col] = pd.to_numeric(df[col], errors='coerce')
  df[col] = df[col].fillna(df[col].median())

# 6. Standardize text columns (trim spaces and unify casing)
for col in ['Order_Status', 'Payment_Method']:
  df[col] = df[col].astype(str).str.strip().str.title()

df['Customer_ID'] = df['Customer_ID'].fillna('Unknown').astype(str).str.strip()

# 7. Parse dates and fix Ship_Date if earlier than Order_Date
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df['Ship_Date'] = pd.to_datetime(df['Ship_Date'], errors='coerce')
df.loc[df['Ship_Date'] < df['Order_Date'], 'Ship_Date'] = pd.NaT

# 8. Final check and export
print('\nShape after cleaning:', df.shape)
print('\nMissing values after cleaning:')
print(df.isna().sum())
display(df.head(10))

df.to_csv('cleaned_new_daily_batch.csv', index=False)
print('\nDone! Saved to cleaned_new_daily_batch.csv')

Shape before cleaning: (333, 11)

Missing values before cleaning:
Order_ID           0
Order_Date         0
Ship_Date         14
Customer_ID        4
Product_ID         0
Quantity           0
Unit_Price         0
Discount           0
Shipping_Cost      9
Order_Status       0
Payment_Method     0
dtype: int64

Shape after cleaning: (329, 11)

Missing values after cleaning:
Order_ID           0
Order_Date         0
Ship_Date         17
Customer_ID        0
Product_ID         0
Quantity           0
Unit_Price         0
Discount           0
Shipping_Cost      0
Order_Status       0
Payment_Method     0
dtype: int64


,Order_ID,Order_Date,Ship_Date,Customer_ID,Product_ID,Quantity,Unit_Price,Discount,Shipping_Cost,Order_Status,Payment_Method
0,NB000090,2025-11-01,2025-11-11,C0070,P007,1,332.30,0.00,25.0,Delivered,Wallet
1,NB000034,2025-11-04,2025-11-12,C0128,P018,2,160.99,0.10,25.0,Delivered,Wallet
2,NB000184,2025-11-01,2025-11-06,C0128,P021,1,757.22,0.00,35.0,Delivered,Cash
3,NB000197,2025-11-04,2025-11-09,C0007,P029,5,63.83,0.00,45.0,Shipped,Wallet
4,NB000108,2025-11-03,2025-11-07,C0133,P028,1,57.93,0.10,45.0,Delivered,Wallet
5,NB000108,2025-11-03,2025-11-07,C0133,P012,1,148.74,0.05,45.0,Delivered,Wallet
6,NB000098,2025-11-01,2025-11-09,C0036,P027,2,752.11,0.00,25.0,Delivered,Online
7,NB000168,2025-11-05,2025-11-08,C0067,P014,1,179.93,0.00,35.0,Cancelled,Card
8,NB000128,2025-11-01,2025-11-05,C0023,P029,1,63.83,0.00,25.0,Delivered,Online
9,NB000180,2025-11-01,2025-11-05,C0098,P008,2,793.91,0.10,25.0,Delivered,Card



Done! Saved to cleaned_new_daily_batch.csv
